# CIFAR-10 Object Recognition
Trains and compares two CNN architectures on the CIFAR-10 dataset.

## 1. Imports & Configuration

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tqdm import tqdm

BATCH_SIZE = 64
EPOCHS = 15
LR = 0.001
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 2. Dataset Statistics

Compute per-channel mean and standard deviation over the CIFAR-10 training set. These values are used to normalize inputs in the next section.

In [ ]:
_raw = torchvision.datasets.CIFAR10(root='.', train=True, download=False,
                                    transform=transforms.ToTensor())
_loader = torch.utils.data.DataLoader(_raw, batch_size=len(_raw), num_workers=2)
_images, _ = next(iter(_loader))  # [50000, 3, 32, 32]

MEAN = tuple(_images.mean(dim=[0, 2, 3]).tolist())
STD  = tuple(_images.std(dim=[0, 2, 3]).tolist())

print(f'MEAN: {tuple(round(v, 4) for v in MEAN)}')
print(f'STD:  {tuple(round(v, 4) for v in STD)}')

## 3. Data Loading & Preprocessing

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# root='.' tells torchvision to look for cifar-10-batches-py/ in the current directory
train_dataset = torchvision.datasets.CIFAR10(root='.', train=True,  download=False, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root='.', train=False, download=False, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train samples: {len(train_dataset):,}  |  Test samples: {len(test_dataset):,}')

## 4. Model Definitions

**SimpleCNN** — 2 convolutional layers, no batch normalization; fast baseline.  
**DeepCNN** — 3 convolutional layers with BatchNorm and Dropout; more capacity.

> If you have saved weights (`simple_cnn.pth` / `deep_cnn.pth`), you can skip sections 5–6 and jump to section 7.

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 32 -> 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                             # 16 -> 8
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 8 * 8, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class DeepCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                              # 32 -> 16
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),                              # 16 -> 8
            nn.Dropout2d(0.3),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


print('SimpleCNN parameters:', sum(p.numel() for p in SimpleCNN().parameters()))
print('DeepCNN  parameters:', sum(p.numel() for p in DeepCNN().parameters()))

SimpleCNN parameters: 1070794
DeepCNN  parameters: 2193674


## 5. Training

In [5]:
def train_model(model, name, epochs=EPOCHS):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    history = {'loss': []}

    print(f'\n=== Training {name} for {epochs} epochs ===')
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        progress = tqdm(train_loader, desc=f'Epoch [{epoch+1}/{epochs}]', leave=True)
        for inputs, labels in progress:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            progress.set_postfix(loss=f'{running_loss / (progress.n + 1):.4f}')
        epoch_loss = running_loss / len(train_loader)
        history['loss'].append(epoch_loss)

    print(f'{name} training complete.')
    return model, history


simple_model, simple_history = train_model(SimpleCNN(), 'SimpleCNN')
deep_model,   deep_history   = train_model(DeepCNN(),   'DeepCNN')


=== Training SimpleCNN for 15 epochs ===


Epoch [15/15]: 100%|██████████| 782/782 [00:30<00:00, 25.85it/s, loss=0.6528]


SimpleCNN training complete.

=== Training DeepCNN for 15 epochs ===


Epoch [15/15]: 100%|██████████| 782/782 [01:15<00:00, 10.35it/s, loss=0.8928]

DeepCNN training complete.


## 6. Save Models

In [ ]:
torch.save(simple_model.state_dict(), 'simple_cnn.pth')
torch.save(deep_model.state_dict(), 'deep_cnn.pth')
print('Models saved to simple_cnn.pth and deep_cnn.pth')

## 7. Load Saved Models (optional — skip if models are already in memory)

Run sections 1–4 first so `SimpleCNN`, `DeepCNN`, and `device` are defined.

In [ ]:
simple_model = SimpleCNN().to(device)
simple_model.load_state_dict(torch.load('simple_cnn.pth', map_location=device))

deep_model = DeepCNN().to(device)
deep_model.load_state_dict(torch.load('deep_cnn.pth', map_location=device))

print('Models loaded from simple_cnn.pth and deep_cnn.pth')

## 8. Evaluation

In [ ]:
def evaluate(model):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            preds = model(inputs).argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels.append(labels)
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    accuracy = (all_preds == all_labels).mean()
    n = len(all_labels)
    ci = 1.96 * np.sqrt(accuracy * (1 - accuracy) / n)
    return accuracy, ci, all_preds, all_labels


simple_acc, simple_ci, simple_preds, simple_labels = evaluate(simple_model)
deep_acc,   deep_ci,   deep_preds,   deep_labels   = evaluate(deep_model)

print(f'{'Model':<12} {'Accuracy':>10}  {'95% CI':>14}')
print('-' * 40)
print(f'{'SimpleCNN':<12} {simple_acc:>9.2%}  ±{simple_ci:.4f}')
print(f'{'DeepCNN':<12} {deep_acc:>9.2%}  ±{deep_ci:.4f}')

## 9. Training Curves

> Requires `simple_history` and `deep_history` from section 5. Skip this section if you loaded models from saved weights.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(simple_history['loss'], label='SimpleCNN', marker='o')
plt.plot(deep_history['loss'],   label='DeepCNN',   marker='s')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Training Loss per Epoch')
plt.legend()
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 10. Normalized Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, preds, labels, title in [
    (axes[0], simple_preds, simple_labels, f'SimpleCNN  ({simple_acc:.2%} ± {simple_ci:.4f})'),
    (axes[1], deep_preds,   deep_labels,   f'DeepCNN    ({deep_acc:.2%} ± {deep_ci:.4f})'),
]:
    cm = confusion_matrix(labels, preds, normalize='true')
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASSES)
    disp.plot(ax=ax, colorbar=True, xticks_rotation=45, values_format='.2f')
    ax.set_title(title)

plt.suptitle('Normalized Confusion Matrices (row = true class)', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

## 11. Per-class Accuracy

In [ ]:
print(f'{'Class':<12} {'SimpleCNN':>10}  {'DeepCNN':>10}')
print('-' * 36)
for i, cls in enumerate(CLASSES):
    mask = simple_labels == i
    s_acc = (simple_preds[mask] == i).mean()
    d_acc = (deep_preds[mask] == i).mean()
    print(f'{cls:<12} {s_acc:>9.2%}  {d_acc:>9.2%}')